# Práctica 3 · Etapa 6 — Extraclase: validación temporal rigurosa, comparación estadística y ampliaciones

**3201 Modelos Inteligentes de Aprendizaje / 529 Redes Neuronales Modulares** · Maestría y Doctorado en Ciencias e Ingeniería (MyDCI) · Facultad de Ingeniería, Arquitectura y Diseño ·
Universidad Autónoma de Baja California · Ciclo 2026-2 · Versión 6

**Profesores:** Dr. Everardo Inzunza González · Dr. Enrique Efrén García Guerrero · Dr. Oscar Adrián Aguirre Castro

**Componente extraclase (Anexo P del manual)** · presupuesto: **≈ 10 h repartidas en 1–2 semanas**

---

La sesión evaluó cada modelo con **una** partición. Un solo número no dice cuánto variaría el resultado con otro origen de pronóstico ni si la diferencia entre dos modelos es real o ruido. Esta etapa añade cuatro piezas: (P1) validación cruzada temporal **anidada** de 5 pliegues con media ± desviación estándar; (P2) la prueba de **Diebold–Mariano** para comparar pronósticos sobre el mismo bloque de prueba, con su diagnóstico; (P3) la ampliación del modelo del video con **rezagos anuales** y validación cruzada temporal; (P4) la aplicación del flujo a **su propio conjunto de datos**. La P5 (reproducción del artículo ancla) se especifica en el Anexo P.

| Necesita | Produce |
|---|---|
| artefactos de las Etapas 2–5 · `datos/PJME_hourly.csv` · su propio conjunto de datos (P4) | Tablas J, K y L · figuras F14 a F16 (van en el reporte) · `tablas/tabla_J_cv_anidada.csv`, `tabla_K_dm.csv` |

**Cómo leer este cuaderno.** Cada bloque abre con una celda de texto que explica *qué* se hace y *por qué*. Dentro del
código, el comentario a la derecha de cada instrucción dice para qué sirve esa línea. Las salidas marcadas con
**`← Anote`** son las que usted copia en la tabla indicada del manual; el resto sólo se ejecuta y se observa.
Ejecute las celdas **en orden**; si cambia algo, vuelva a ejecutar desde esa celda hacia abajo.

---
## Bloque 0 · Puesta en marcha

Esta celda se ejecuta **siempre primero**, en cualquier entorno. Detecta si está en Google Colab, en el JupyterHub del
laboratorio o en un Jupyter local; localiza la carpeta de la práctica (`P3_SeriesTiempo_v6`, la que descargó de
Classroom); crea `resultados/` con sus subcarpetas sin borrar nada; instala sólo lo que falte; fija la semilla e imprime
las versiones con que se ejecutó. Reproducir un resultado empieza por saber con qué se obtuvo.

> **En Colab**, cada cuaderno corre en su propia máquina virtual: los archivos que un cuaderno guarda sólo los ve el
> siguiente si viven en **Google Drive**. Por eso la carpeta de la práctica se sube a *Mi unidad* y esta celda la monta.

In [ ]:
# ── Bloque 0 · Puesta en marcha (Etapa 6) ─────────────────────────────────────────────
import sys, os, json, time, hashlib, warnings, subprocess, importlib  # utilidades estándar de Python
from pathlib import Path                                   # rutas que funcionan igual en Windows, Linux y Colab
T0 = time.perf_counter()                                   # cronómetro de este cuaderno
warnings.filterwarnings("ignore", category=FutureWarning)  # oculta sólo avisos de versiones futuras

try:
    import google.colab                                    # este módulo sólo existe dentro de Google Colab
    EN_COLAB = True
except ImportError:
    EN_COLAB = False

CARPETA_PRACTICA = "P3_SeriesTiempo_v6"                    # la carpeta que descargó de Classroom
if EN_COLAB:
    from google.colab import drive                         # acceso a Google Drive desde Colab
    drive.mount("/content/drive")                          # la primera vez pide autorización
    RAIZ_DRIVE = Path("/content/drive/MyDrive")            # «Mi unidad»
    candidatas = [RAIZ_DRIVE / CARPETA_PRACTICA, RAIZ_DRIVE / "Colab Notebooks" / CARPETA_PRACTICA]
    candidatas += sorted(RAIZ_DRIVE.glob(f"*/{CARPETA_PRACTICA}"))  # también un nivel más abajo
else:
    aqui = Path.cwd().resolve()                            # carpeta desde donde se abrió el cuaderno
    candidatas = [aqui, *aqui.parents][:4]                 # el cuaderno vive en notebooks/: se busca hacia arriba
    candidatas += [p / CARPETA_PRACTICA for p in candidatas]
BASE = next((c for c in candidatas if (c / "datos").is_dir() and (c / "notebooks").is_dir()), None)
if BASE is None:
    raise FileNotFoundError("No encuentro la carpeta de la práctica (con datos/ y notebooks/). "
                            "Revise la Sección 4.3 del manual: en Colab súbala completa a «Mi unidad».")
DATOS, RES = BASE / "datos", BASE / "resultados"           # entradas y salidas de la práctica
FIG, TAB, ART = RES / "figuras", RES / "tablas", RES / "artefactos"  # figuras · tablas · archivos intermedios
for carpeta in (FIG, TAB, ART):
    carpeta.mkdir(parents=True, exist_ok=True)             # se crean si faltan; nunca se borra nada

# Bibliotecas: se instala sólo lo que falte (en Colab casi todo viene preinstalado).
NECESARIAS = {"numpy": "numpy", "pandas": "pandas", "sklearn": "scikit-learn", "xgboost": "xgboost",
               "matplotlib": "matplotlib", "seaborn": "seaborn", "statsmodels": "statsmodels",
               "openpyxl": "openpyxl", "scipy": "scipy"}
for modulo, paquete in NECESARIAS.items():
    try:
        importlib.import_module(modulo)                    # ¿ya está instalado?
    except ImportError:
        print("Instalando", paquete, "…")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", paquete], check=True)

import numpy as np                                         # cálculo numérico con arreglos
import pandas as pd                                        # tablas de datos indexadas por fecha
import matplotlib                                          # motor de gráficas
import matplotlib.pyplot as plt                            # interfaz de gráficas
import sklearn, xgboost as xgb                             # scikit-learn y XGBoost
import seaborn, statsmodels, scipy                         # sólo para registrar sus versiones aquí
from scipy import stats                                    # distribución t para la prueba de Diebold–Mariano

def _ver(v):
    return tuple(int("".join(ch for ch in p if ch.isdigit()) or 0) for p in v.split(".")[:2])
assert _ver(sklearn.__version__) >= (1, 4), "Se necesita scikit-learn ≥ 1.4 (root_mean_squared_error)."
assert _ver(xgb.__version__) >= (2, 0), "Se necesita XGBoost ≥ 2.0 (early_stopping_rounds en el constructor)."

SEMILLA = 42                                               # una sola semilla para toda la práctica
np.random.seed(SEMILLA)                                    # fija el azar de numpy
pd.set_option("display.width", 160)                        # tablas anchas sin cortes de renglón
pd.set_option("display.max_columns", 30)
COLORES = {"real": "#15241E", "XGBoost": "#17795C", "SVR": "#D9A21B", "Random Forest": "#9E2B25",
            "Ingenuo": "#9AA7A1", "Ingenuo estacional": "#5E6E67", "Climatología": "#165542"}
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3, "axes.spines.top": False,
                     "axes.spines.right": False, "font.size": 10})  # estilo uniforme de las figuras

def guardar(fig, nombre):
    """Guarda la figura en resultados/figuras/ (PNG, 150 dpi) para insertarla en el manual."""
    ruta = FIG / f"{nombre}.png"
    fig.savefig(ruta, dpi=150, bbox_inches="tight")
    print("Figura guardada →", ruta.relative_to(BASE))

def _serializable(v):
    """Convierte tipos de numpy y pandas a tipos de Python; los no finitos (NaN, ±inf) quedan como None."""
    if isinstance(v, dict):
        return {str(k): _serializable(x) for k, x in v.items()}
    if isinstance(v, (list, tuple)):
        return [_serializable(x) for x in v]
    if isinstance(v, (bool, np.bool_)):
        return bool(v)
    if isinstance(v, (int, np.integer)):
        return int(v)
    if isinstance(v, (float, np.floating)):
        return float(v) if np.isfinite(v) else None
    if isinstance(v, pd.Timestamp):
        return v.isoformat()
    return v

def registrar(clave, valor):
    """Añade un resultado a resultados/registro_P3.json (se acumula entre cuadernos; JSON estricto, sin NaN)."""
    ruta = RES / "registro_P3.json"
    reg = json.loads(ruta.read_text(encoding="utf-8")) if ruta.exists() else {}
    reg[clave] = _serializable(valor)
    ruta.write_text(json.dumps(reg, ensure_ascii=False, indent=1, allow_nan=False), encoding="utf-8")

def md5(ruta):
    return hashlib.md5(Path(ruta).read_bytes()).hexdigest()  # huella del archivo

if EN_COLAB:
    ENTORNO = "Google Colab"
elif "JUPYTERHUB_USER" in os.environ:
    ENTORNO = "JupyterHub"                                 # sesión abierta en el JupyterHub del laboratorio
elif "ai_env" in sys.executable:
    ENTORNO = "Kernel ai_env del laboratorio"              # el mismo kernel, fuera de una sesión del Hub
else:
    ENTORNO = "Jupyter local"
VERSIONES = {"python": sys.version.split()[0], "numpy": np.__version__, "pandas": pd.__version__,
              "scikit-learn": sklearn.__version__, "xgboost": xgb.__version__, "matplotlib": matplotlib.__version__,
              "seaborn": seaborn.__version__, "statsmodels": statsmodels.__version__, "scipy": scipy.__version__}
registrar("etapa6_entorno", {"entorno": ENTORNO, **VERSIONES})
print("Entorno        :", ENTORNO)                         # ← Anote (Tabla A)
print("Carpeta base   :", BASE)
for k, v in VERSIONES.items():
    print(f"{k:<15}:", v)                                  # ← Anote (Tabla A): Python, scikit-learn, XGBoost y pandas

In [ ]:
from sklearn.model_selection import train_test_split, TimeSeriesSplit, GridSearchCV, ParameterGrid
from sklearn.base import clone
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.compose import TransformedTargetRegressor
from sklearn.svm import SVR
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, root_mean_squared_error, mean_absolute_error, r2_score
from xgboost import XGBRegressor

for nombre in ("caracteristicas_2038.csv", "serie_2038_limpia.csv", "particion_2038.json", "predicciones_2038.csv",
               "lineas_base_prueba_2038.csv"):
    if not (ART / nombre).exists():
        raise FileNotFoundError(f"Falta {nombre}: ejecute primero las Etapas 2 a 5.")
tabla = pd.read_csv(ART / "caracteristicas_2038.csv", index_col="FECHA", parse_dates=True)  # de la Etapa 2
y = tabla.pop("y"); X = tabla
X_train, X_tmp, y_train, y_tmp = train_test_split(X, y, test_size=0.30, shuffle=False)  # misma partición
X_val, X_test, y_val, y_test = train_test_split(X_tmp, y_tmp, test_size=0.50, shuffle=False)
guardada = json.loads((ART / "particion_2038.json").read_text(encoding="utf-8"))
for nombre, parte in {"entrenamiento": y_train, "validación": y_val, "prueba": y_test}.items():
    assert [f"{t:%Y-%m}" for t in parte.index] == guardada[nombre]["fechas"], f"Bloque «{nombre}» distinto al de la Etapa 3"
X_trval, y_trval = pd.concat([X_train, X_val]), pd.concat([y_train, y_val])  # datos de desarrollo (sin la prueba)
serie_regular = pd.read_csv(ART / "serie_2038_limpia.csv", index_col="FECHA", parse_dates=True)  # rejilla mensual + bandera
MODELOS = {                                    # mismos modelos y rejillas de la Etapa 4
    "XGBoost": (XGBRegressor(objective="reg:squarederror", tree_method="hist", random_state=SEMILLA, n_jobs=-1),
                {"n_estimators": [100, 300], "max_depth": [2, 3, 5], "learning_rate": [0.03, 0.1]}),
    "SVR": (TransformedTargetRegressor(regressor=make_pipeline(StandardScaler(), SVR(kernel="rbf")),
                                       transformer=StandardScaler()),
            {"regressor__svr__C": [0.1, 1, 10], "regressor__svr__gamma": ["scale", 0.01, 0.1],
             "regressor__svr__epsilon": [0.05, 0.1, 0.3]}),
    "Random Forest": (RandomForestRegressor(random_state=SEMILLA, n_jobs=-1),
                      {"n_estimators": [200, 500], "max_depth": [3, 5, None], "min_samples_leaf": [1, 5]}),
}
NOMBRES = list(MODELOS)

def escala_mase(serie_regular, hasta, m=12):
    """MAE del ingenuo estacional por calendario (Etapa 5): pares (t, t−m) observados con t ≤ hasta → (escala, n pares)."""
    s = serie_regular["PRECIP_mm"]
    observado = ~serie_regular["imputado"].astype(bool)
    diferencia = (s - s.shift(m)).loc[:hasta]  # y(t) − y(t−m), alineado por calendario
    validos = (observado & observado.shift(m, fill_value=False)).loc[:hasta]  # ambos meses observados
    return float(diferencia[validos].abs().mean()), int(validos.sum())

print("Datos de desarrollo:", X_trval.shape, "· prueba reservada:", X_test.shape)

---
## P1 · Validación cruzada temporal anidada, k = 5 (Tabla J)

**Qué se estima.** El desempeño esperado de *todo el procedimiento* (elegir hiperparámetros + ajustar) cuando se
pronostica desde orígenes distintos. **Cómo.** Un bucle externo `TimeSeriesSplit(n_splits=5)` sobre los datos de
desarrollo (entrenamiento + validación; la prueba sigue sin tocarse). En cada pliegue externo, `GridSearchCV` con un
`TimeSeriesSplit(n_splits=3)` interno elige los hiperparámetros **usando sólo el pasado de ese pliegue**; el modelo
elegido se evalúa en el bloque siguiente. Así ningún hiperparámetro se elige mirando los datos con que se evalúa. La
escala del MASE se calcula en cada pliegue **por calendario** y sólo con el pasado del pliegue (`escala_mase`, con
`hasta` = último mes de ajuste).

> **Criterio de aceptación.** Media ± DE de RMSE, MAE, MASE y habilidad frente a la climatología para los tres modelos
> y las tres líneas base, con las líneas base calculadas **en cada pliegue** sólo con su entrenamiento. Reporte también
> los hiperparámetros elegidos en cada pliegue: si cambian mucho, dígalo. Anote la duración de la celda.

In [ ]:
externo, interno = TimeSeriesSplit(n_splits=5), TimeSeriesSplit(n_splits=3)  # anidamiento temporal
registros, elegidos = [], []
t0 = time.perf_counter()
for k, (itr, ite) in enumerate(externo.split(X_trval), start=1):
    Xa, ya, Xb, yb = X_trval.iloc[itr], y_trval.iloc[itr], X_trval.iloc[ite], y_trval.iloc[ite]  # ajuste | evaluación
    escala, n_pares = escala_mase(serie_regular, hasta=ya.index[-1])  # escala del MASE con el pasado del pliegue
    clim = ya.groupby(ya.index.month).mean().reindex(Xb.index.month).values  # climatología del pliegue (12 medias)
    mse_clim = mean_squared_error(yb, clim)                # referencia de la habilidad en este pliegue
    preds = {"Ingenuo": Xb["lag_1"].values, "Ingenuo estacional": Xb["lag_12"].values, "Climatología": clim}
    for n, (est, rej) in MODELOS.items():
        gs = GridSearchCV(clone(est), rej, cv=interno, scoring="neg_root_mean_squared_error", n_jobs=-1).fit(Xa, ya)  # selección interna
        preds[n] = gs.predict(Xb)                          # el mejor de la rejilla, reajustado con todo el pliegue
        elegidos.append({"pliegue": k, "modelo": n, **{p.split("__")[-1]: v for p, v in gs.best_params_.items()}})
    for n, p in preds.items():
        registros.append({"pliegue": k, "método": n, "n_ajuste": len(itr), "n_eval": len(ite), "desde": f"{Xb.index[0]:%Y-%m}",
                          "RMSE": root_mean_squared_error(yb, p),  # error cuadrático medio, raíz
                          "MAE": mean_absolute_error(yb, p),  # error absoluto medio
                          "MASE": mean_absolute_error(yb, p) / escala,  # MAE relativo al ingenuo estacional del pasado
                          "Hab_clim": 1 - mean_squared_error(yb, p) / mse_clim})  # habilidad frente a la climatología
    print(f"pliegue {k}: ajuste {len(itr)} meses → evaluación {Xb.index[0]:%Y-%m}…{Xb.index[-1]:%Y-%m} "
          f"({len(ite)} meses) · escala MASE {escala:.2f} mm ({n_pares} pares)")
cv = pd.DataFrame(registros)
t_p1 = time.perf_counter() - t0
print(f"Duración de la validación anidada: {t_p1:.1f} s")  # ← Anote (Tabla J): fila de duración
tabla_J = cv.groupby("método")[["RMSE", "MAE", "MASE", "Hab_clim"]].agg(["mean", "std"]).round(3)
tabla_J.to_csv(TAB / "tabla_J_cv_anidada.csv")
tabla_J.sort_values(("RMSE", "mean"))                      # ← Anote (Tabla J): media y DE de cada método

In [ ]:
pd.DataFrame(elegidos).set_index(["modelo", "pliegue"]).sort_index()  # ¿son estables los hiperparámetros?

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
for n in ["Climatología", "Ingenuo estacional"] + NOMBRES:
    d = cv[cv["método"] == n]
    ax.plot(d["pliegue"], d["RMSE"], "o-", label=n, color=COLORES.get(n, "gray"))  # una línea por método
ax.set(title="RMSE por pliegue de la validación cruzada anidada", xlabel="Pliegue externo (más reciente →)",
       ylabel="RMSE (mm)", xticks=range(1, 6))
ax.legend(fontsize=8)
guardar(fig, "F14_est2038_cv_por_pliegue")  # → Figura F14 (reporte)
plt.show()

---
## P2 · ¿La diferencia es real? Prueba de Diebold–Mariano (Tabla K)

Dos pronósticos cuyos RMSE difieren en décimas de milímetro sobre el bloque de prueba pueden no ser distinguibles. La
prueba de Diebold y Mariano (1995) compara las **pérdidas** de dos pronósticos sobre las mismas fechas: con
$d_t = L(e_{1t}) - L(e_{2t})$, contrasta $H_0: E[d_t] = 0$. Se aplica con la corrección para muestras pequeñas de
Harvey, Leybourne y Newbold (1997) y la distribución $t$ con $n-1$ grados de libertad. Con horizonte $h = 1$ la
varianza de $\bar d$ se estima con la autocovarianza de orden cero, lo que **supone** que los $d_t$ no están
autocorrelacionados. Esa suposición se **diagnostica** y se **relaja**:

- **Diagnóstico.** Prueba de Ljung–Box sobre $d_t$ con 12 rezagos (`statsmodels`): un valor *p* pequeño indica
  autocorrelación en el diferencial y, con ella, una varianza de $\bar d$ mal estimada por el DM-HLN con $h = 1$.
- **Sensibilidad.** Un estadístico $t$ con la varianza **HAC de Newey–West** (núcleo de Bartlett, con
  $L = \lfloor 4\,(n/100)^{2/9} \rfloor$ rezagos), que admite autocorrelación y heterocedasticidad en $d_t$. No es el
  mismo estadístico que el DM-HLN: **no** lleva el factor de corrección de Harvey et al., y su valor *p* (`p_HAC`) se
  reporta **sin** corregir por multiplicidad; la corrección de Holm se aplica sólo a `p`. Si la conclusión cambia entre
  `p` y `p_HAC`, la decisión depende del supuesto y hay que decirlo.

Dos advertencias más. **No rechazar $H_0$ no es demostrar equivalencia**: con un bloque de prueba de poco más de una
década la prueba tiene poca potencia
para diferencias pequeñas, y «no significativo» se lee como «no se pudo distinguir», nunca como «iguales». Y el bloque
de prueba tiene al menos un **hueco de calendario** (un mes con objetivo imputado que se retiró de la evaluación; el
cuaderno lo imprime): la serie $d_t$ se trata como consecutiva aunque no lo sea del todo, lo que afecta más a los
estadísticos que usan rezagos (Ljung–Box, HAC) que al DM-HLN con $h = 1$.

> **Las comparaciones se fijan antes de mirar la prueba.** Elegir «el mejor modelo de la prueba» y después probar si
> supera a la línea base es una selección *a posteriori* que vicia el valor *p*; además, el orden entre modelos con
> errores parecidos puede cambiar entre entornos (las corridas difieren en varias bibliotecas a la vez). Por eso las
> siete comparaciones están decididas de antemano: cada modelo frente a la climatología y frente al ingenuo
> estacional, más XGBoost frente a Random Forest.
>
> **Criterio de aceptación.** Tabla K en formato largo: una fila por comparación y pérdida (7 × 2 = 14 filas) con
> $\bar d$, estadístico DM-HLN, valor *p*, *p* corregido por Holm dentro de cada pérdida, *p* con varianza HAC (sin
> corregir) y *p* de Ljung–Box del diferencial; al pie, *n* y el número de rezagos $L$ de Newey–West. Interprete el
> signo ($\bar d < 0$: el primer pronóstico tiene menor pérdida), diga qué conclusión cambia al corregir por Holm y qué
> comparaciones tienen un diferencial autocorrelacionado.

In [ ]:
from statsmodels.stats.diagnostic import acorr_ljungbox

def diebold_mariano(e1, e2, h=1, perdida="cuadrática"):
    """Prueba DM con corrección de Harvey, Leybourne y Newbold (1997), más Ljung–Box y varianza HAC del diferencial."""
    e1, e2 = np.asarray(e1, float), np.asarray(e2, float)
    d = e1 ** 2 - e2 ** 2 if perdida == "cuadrática" else np.abs(e1) - np.abs(e2)  # diferencial de pérdida
    n, dbar = len(d), d.mean()
    gam = lambda k: np.sum((d[k:] - dbar) * (d[:n - k] - dbar)) / n  # autocovarianza de orden k
    var_dbar = (gam(0) + 2 * sum(gam(k) for k in range(1, h))) / n  # varianza de d̄ con h−1 rezagos (DM clásico)
    ajuste = np.sqrt((n + 1 - 2 * h + h * (h - 1) / n) / n)  # corrección HLN para muestras pequeñas
    estad = dbar / np.sqrt(var_dbar) * ajuste
    p = 2 * stats.t.sf(abs(estad), df=n - 1)     # valor p bilateral, t con n−1 g.l.
    L = int(np.floor(4 * (n / 100) ** (2 / 9)))  # rezagos de Newey–West
    var_hac = (gam(0) + 2 * sum((1 - k / (L + 1)) * gam(k) for k in range(1, L + 1))) / n  # núcleo de Bartlett
    p_hac = 2 * stats.t.sf(abs(dbar / np.sqrt(var_hac)), df=n - 1)  # t con varianza HAC, SIN el factor HLN; p sin corregir por Holm
    p_lb = float(acorr_ljungbox(d, lags=[12], return_df=True)["lb_pvalue"].iloc[0])  # ¿d_t autocorrelacionado?
    return {"d_medio": dbar, "DM_HLN": estad, "p": p, "p_HAC": p_hac, "L_NW": L, "p_LjungBox": p_lb, "n": n}

pred = pd.read_csv(ART / "predicciones_2038.csv", index_col="FECHA", parse_dates=True)  # pronósticos de prueba (Etapa 4)
lbp = pd.read_csv(ART / "lineas_base_prueba_2038.csv", index_col="FECHA", parse_dates=True)  # líneas base (Etapa 5)
assert (lbp.index == pred.index).all(), "Las líneas base de prueba no están alineadas con las predicciones"
for b in ["Climatología", "Ingenuo estacional"]:
    pred[b] = lbp[b]
err = {n: pred["real"] - pred[n] for n in NOMBRES + ["Climatología", "Ingenuo estacional"]}  # errores de prueba
pares = [(n, b) for n in NOMBRES for b in ("Climatología", "Ingenuo estacional")] + [("XGBoost", "Random Forest")]
huecos = pd.date_range(pred.index[0], pred.index[-1], freq="MS").difference(pred.index)
print("Fechas de prueba:", len(pred), "· huecos de calendario dentro de la prueba:", [f"{t:%Y-%m}" for t in huecos])

def holm(p):
    """Corrección de Holm–Bonferroni para una familia de valores p."""
    p = np.asarray(p, float); orden = np.argsort(p); m = len(p)
    ajustado, acumulado = np.empty(m), 0.0
    for rango, i in enumerate(orden):
        acumulado = max(acumulado, min(1.0, (m - rango) * p[i]))  # nunca decrece
        ajustado[i] = acumulado
    return ajustado

filas = []
for perdida in ("cuadrática", "absoluta"):
    bloque = []
    for a, b in pares:
        bloque.append({"comparación": f"{a} vs {b}", "pérdida": perdida, **diebold_mariano(err[a], err[b], perdida=perdida)})
    for fila, ph in zip(bloque, holm([f["p"] for f in bloque])):
        fila["p_Holm"] = ph                      # corregido dentro de la misma pérdida (7 pruebas)
    filas += bloque
tabla_K = pd.DataFrame(filas)[["comparación", "pérdida", "n", "d_medio", "DM_HLN", "p", "p_Holm", "p_HAC", "L_NW", "p_LjungBox"]]
tabla_K.to_csv(TAB / "tabla_K_dm.csv", index=False)
tabla_K.round(4)                                 # ← Anote (Tabla K: las 14 filas; n y L_NW al pie)

---
## P3 · Regreso al consumo eléctrico: rezagos anuales y validación cruzada temporal

El modelo del video (Etapa 1) sólo conoce el calendario y pronostica desde un origen fijo. La pregunta natural es si
añadir **el consumo de la misma hora hace uno, dos y tres años** (364, 728 y 1 092 días: múltiplos de 7, para caer en
el mismo día de la semana) mejora el pronóstico, y si la mejora se sostiene en varios orígenes. Se usa
`TimeSeriesSplit(n_splits=5, test_size=24·365, gap=24)`: cinco años de prueba consecutivos, con un día de separación
entre ajuste y evaluación. La detención temprana usa el último 10 % de cada ajuste, **nunca** el año que se evalúa.

**Protocolo de evaluación: secuencial.** Cada bloque de prueba dura 365 días y el rezago más corto es de 364: en las
**últimas horas de cada bloque**, el rezago de 364 días cae dentro del mismo bloque que se evalúa, es decir, usa como
característica un consumo que, con un origen fijo al inicio del bloque, aún no se habría observado. El modelo con
rezagos supone, por tanto, que al emitir cada pronóstico se conoce el consumo de hace 364 días: ya no pronostica desde un
origen fijo. La comparación con el modelo de sólo calendario sigue siendo válida sobre las mismas horas, pero compara
dos **protocolos de información distintos**: declárelo en su reporte, igual que se hizo en la Etapa 1 con los ingenuos.

> **Criterio de aceptación.** RMSE por pliegue y media ± DE con y sin rezagos sobre los mismos pliegues; árboles usados
> para predecir (`best_iteration + 1`) y rondas entrenadas (`num_boosted_rounds()`) en cada caso; la duración de la
> celda (fila de la Tabla L); una frase que diga si la mejora es consistente en los cinco pliegues y otra que declare
> el protocolo de información.

In [ ]:
ruta_pjme = DATOS / "PJME_hourly.csv"
if not ruta_pjme.exists():
    raise FileNotFoundError("Falta datos/PJME_hourly.csv (viene en el paquete de Classroom; ver Etapa 1).")
pjme = (pd.read_csv(ruta_pjme, parse_dates=["Datetime"]).set_index("Datetime").sort_index())  # orden cronológico
pjme = pjme[~pjme.index.duplicated(keep="first")]  # quita marcas de hora repetidas (Etapa 1)

def calendario(df):
    """Las características del video: todas derivadas de la fecha y hora."""
    df = df.copy()
    idx = df.index
    df["hora"], df["dia_semana"], df["trimestre"], df["mes"] = idx.hour, idx.dayofweek, idx.quarter, idx.month
    df["anio"], df["dia_anio"], df["dia_mes"] = idx.year, idx.dayofyear, idx.day
    df["semana_anio"] = idx.isocalendar().week.astype(int).values  # reemplazo de .weekofyear (eliminado en pandas 2)
    return df

def rezagos(df, serie):
    """Consumo de la misma hora hace 364, 728 y 1092 días (NaN si no existe ese instante)."""
    df = df.copy()
    for k, dias in enumerate((364, 728, 1092), start=1):
        df[f"rezago_{k}"] = (df.index - pd.Timedelta(days=dias)).map(serie)  # busca el valor de esa fecha exacta
    return df

datos = rezagos(calendario(pjme), pjme["PJME_MW"])
CAL = ["hora", "dia_semana", "trimestre", "mes", "anio", "dia_anio", "dia_mes", "semana_anio"]
CON_REZAGOS = CAL + ["rezago_1", "rezago_2", "rezago_3"]
tss = TimeSeriesSplit(n_splits=5, test_size=24 * 365, gap=24)  # cinco años de prueba de 8 760 horas
p3 = []
t0 = time.perf_counter()
for k, (itr, ite) in enumerate(tss.split(datos), start=1):
    ajuste, prueba = datos.iloc[itr], datos.iloc[ite]
    corte = int(len(ajuste) * 0.9)                 # el último 10 % del ajuste vigila la detención temprana
    for etiqueta, cols in [("sólo calendario", CAL), ("calendario + rezagos", CON_REZAGOS)]:
        m = XGBRegressor(n_estimators=1000, early_stopping_rounds=50, learning_rate=0.01, max_depth=3,
                         objective="reg:squarederror", tree_method="hist", random_state=SEMILLA, n_jobs=-1)
        m.fit(ajuste[cols].iloc[:corte], ajuste["PJME_MW"].iloc[:corte],
              eval_set=[(ajuste[cols].iloc[corte:], ajuste["PJME_MW"].iloc[corte:])], verbose=False)
        p3.append({"pliegue": k, "prueba_desde": f"{prueba.index[0]:%Y-%m-%d}", "variante": etiqueta,
                   "arboles_usados": int(m.best_iteration) + 1,  # los que predicen
                   "rondas": int(m.get_booster().num_boosted_rounds()),  # los que se entrenaron antes de parar
                   "n_eval": int(len(ite)),
                   "RMSE_MW": root_mean_squared_error(prueba["PJME_MW"], m.predict(prueba[cols]))})
p3 = pd.DataFrame(p3)
t_p3 = time.perf_counter() - t0
print(f"Duración de P3: {t_p3:.1f} s")             # ← Anote (Tabla L): fila de duración de P3
p3.pivot(index=["pliegue", "prueba_desde"], columns="variante", values=["RMSE_MW", "arboles_usados", "rondas"]).round(1)

In [ ]:
resumen_p3 = p3.groupby("variante")["RMSE_MW"].agg(["mean", "std"]).round(1)
print(resumen_p3)                    # ← Anote (Tabla L): media ± DE de cada variante
fig, ax = plt.subplots(figsize=(8, 3.8))
for etiqueta, color in [("sólo calendario", "#5E6E67"), ("calendario + rezagos", "#17795C")]:
    d = p3[p3["variante"] == etiqueta]
    ax.plot(d["pliegue"], d["RMSE_MW"], "o-", color=color, label=etiqueta)
ax.set(title="PJME · RMSE por año de prueba (TimeSeriesSplit, 5 pliegues)", xlabel="Pliegue", ylabel="RMSE (MW)",
       xticks=range(1, 6))
ax.legend()
guardar(fig, "F15_pjme_cv_rezagos")  # → Figura F15 (reporte)
plt.show()

---
## P4 · Su propio conjunto de datos

Elija una serie de tiempo **pública, con licencia declarada**, frecuencia regular y al menos 300 observaciones (Kaggle,
UCI, portales de datos abiertos, el SMN o la CONAGUA). Mientras `RUTA_PROPIA` valga `None`, la celda corre una
**demostración** con el consumo PJME agregado a totales diarios (periodo estacional 7), para que vea el flujo completo
funcionando. Para la demostración basta con la celda de configuración; **para su serie, no**: tendrá que leer el
archivo con sus propias columnas, decidir el tratamiento de los huecos y adaptar lo que su serie exija.

**Qué hace `evaluar_serie`, y qué no.** Conserva la **rejilla regular** de la frecuencia declarada (`asfreq`) hasta
haber construido todos los rasgos, de modo que `shift(k)` sea siempre «hace *k* periodos de calendario» aunque la serie
tenga huecos (los huecos se cuentan y se imprimen; las filas incompletas se descartan al final). La fase del ciclo
estacional sale del **calendario real** del índice (día de la semana si `m = 7` con frecuencia diaria; mes si `m = 12`
con frecuencia mensual; hora si `m = 24` con frecuencia horaria), no de la posición en la tabla. Calcula tres líneas
base —ingenuo `lag_1`, ingenuo estacional `lag_m` y **climatología estacional** (media de cada fase del ciclo con los
datos de ajuste)— y tres modelos elegidos **en validación** con la rejilla de la Etapa 4; la prueba se mira una vez.
Reporta RMSE, MAE, MASE (escala por calendario con los datos de ajuste), R² y MAPE (`n/a` si hay ceros en la prueba).
**No hace** la validación anidada: ésa la implementa usted adaptando el código de P1 a su serie, y es parte del
criterio de aceptación.

> **Criterio de aceptación.** (1) Fuente, licencia, frecuencia, periodo, número de observaciones y número de huecos de
> su serie, con la decisión sobre los huecos declarada; (2) justificación escrita del periodo estacional `m`, de los
> rezagos y del horizonte; (3) tabla con las tres líneas base y los tres modelos en la prueba (RMSE, MAE, MASE, R² y,
> si procede, MAPE), más la validación anidada de P1 aplicada a su serie; (4) una figura de serie real frente a
> pronóstico con el modelo elegido en validación. La función `evaluar_serie` es un punto de partida: puede y debe
> modificarla si su serie lo exige (transformaciones, variables exógenas, un horizonte mayor), declarándolo.

In [ ]:
RUTA_PROPIA = None                                      # p. ej. DATOS / "mi_serie.csv"; None = demostración con PJME diario
COLUMNA_FECHA, COLUMNA_VALOR = "fecha", "valor"         # nombres de columnas de SU archivo
FRECUENCIA, M_ESTACIONAL = "D", 7                       # "D" diaria, "MS" mensual, "h" horaria · periodo estacional
REZAGOS = (1, 2, 7, 14)                                 # rezagos en pasos de su frecuencia (justifíquelos)

if RUTA_PROPIA is None:
    propia = pjme["PJME_MW"].resample("D").sum(min_count=24)  # total diario; un día con horas faltantes queda NaN
    propia = propia.loc["2010":].asfreq(FRECUENCIA)     # tramo reciente, para que la demostración sea rápida
    print("Demostración: PJME diario (MWh por día),", len(propia), "días en la rejilla ·",
          int(propia.isna().sum()), "días incompletos (quedan como huecos)")
else:
    crudo = pd.read_csv(RUTA_PROPIA, parse_dates=[COLUMNA_FECHA])
    propia = crudo.set_index(COLUMNA_FECHA)[COLUMNA_VALOR].sort_index().asfreq(FRECUENCIA)  # rejilla regular
    print("Su serie:", len(propia), "observaciones en la rejilla ·", int(propia.isna().sum()), "faltantes")

def fase_estacional(indice, m):
    """Posición en el ciclo estacional leída del calendario real del índice (no de la posición en la tabla)."""
    if m == 7 and indice.freqstr and indice.freqstr.startswith("D"):
        return indice.dayofweek                         # 0 = lunes … 6 = domingo
    if m == 12 and indice.freqstr and indice.freqstr.startswith("M"):
        return indice.month - 1                         # 0 = enero … 11 = diciembre
    if m == 24 and indice.freqstr and indice.freqstr.lower().startswith("h"):
        return indice.hour                              # 0 … 23
    pasos = (indice - indice[0]) // (indice[1] - indice[0])  # otros casos: pasos de calendario desde el origen
    return np.asarray(pasos) % m

def evaluar_serie(s, m, rezagos):
    """Rasgos sobre la rejilla regular → partición 70/15/15 → selección en validación → líneas base y modelos en prueba."""
    s = s.asfreq(s.index.freqstr or pd.infer_freq(s.index))  # garantiza la rejilla regular (los huecos siguen como NaN)
    Xs = pd.DataFrame(index=s.index)
    for k in sorted(set(rezagos) | {1, m}):
        Xs[f"lag_{k}"] = s.shift(k)                     # «hace k periodos de calendario», gracias a la rejilla
    Xs["media_m"] = s.shift(1).rolling(m, min_periods=m).mean()  # nivel reciente, sin fuga
    fase = fase_estacional(s.index, m)
    Xs["ciclo_sin"], Xs["ciclo_cos"] = np.sin(2 * np.pi * fase / m), np.cos(2 * np.pi * fase / m)  # calendario cíclico
    Xs["fase"] = fase
    d = Xs.assign(y=s).dropna()                         # ahora sí: se descartan filas con algún rasgo u objetivo faltante
    print(f"Rejilla: {len(s)} instantes · {int(s.isna().sum())} huecos · filas completas: {len(d)}")
    yy, fase_d = d.pop("y"), d.pop("fase")
    a, tmp, ya, ytmp = train_test_split(d, yy, test_size=0.30, shuffle=False)  # 70 % | 30 %
    v, t, yv, yt = train_test_split(tmp, ytmp, test_size=0.50, shuffle=False)  # 15 % | 15 %
    atv, yatv = pd.concat([a, v]), pd.concat([ya, yv])  # datos de ajuste final
    obs_rej = s.notna()
    hasta = yatv.index[-1]
    validos = (obs_rej & obs_rej.shift(m, fill_value=False)).loc[:hasta]  # pares (t, t−m) ambos observados, t ≤ ajuste
    escala = float((s - s.shift(m)).loc[:hasta][validos].abs().mean())  # MASE por calendario con los datos de ajuste
    clim = yatv.groupby(fase_d.loc[yatv.index]).mean()  # climatología estacional: una media por fase del ciclo
    preds = {"Ingenuo": t["lag_1"].values, "Ingenuo estacional": t[f"lag_{m}"].values,
             "Climatología estacional": clim.reindex(fase_d.loc[t.index]).values}
    elegido = {}
    for n, (est, rej) in MODELOS.items():
        mejor = min(((root_mean_squared_error(yv, clone(est).set_params(**p).fit(a, ya).predict(v)), p)
                     for p in ParameterGrid(rej)), key=lambda z: z[0])  # rejilla medida en validación
        elegido[n] = {"RMSE_val": mejor[0], "params": mejor[1]}
        preds[n] = clone(est).set_params(**mejor[1]).fit(atv, yatv).predict(t)  # reentrena y predice la prueba una vez
    hay_ceros = bool((yt == 0).any())
    res = pd.DataFrame({n: {"n": len(yt), "RMSE": root_mean_squared_error(yt, p), "MAE": mean_absolute_error(yt, p),
                            "MASE": mean_absolute_error(yt, p) / escala, "R2": r2_score(yt, p),
                            "MAPE_%": "n/a" if hay_ceros else 100 * np.mean(np.abs((yt.values - p) / yt.values))}
                        for n, p in preds.items()}).T
    modelo_val = min(elegido, key=lambda n: elegido[n]["RMSE_val"])  # el que se «lleva» a la prueba
    return res, yt, preds, modelo_val, elegido, escala

res_p4, yt_p4, preds_p4, elegido_p4, hp_p4, escala_p4 = evaluar_serie(propia, M_ESTACIONAL, REZAGOS)
print("Modelo elegido en validación:", elegido_p4, "· RMSE de validación:",
      {n: round(v["RMSE_val"], 2) for n, v in hp_p4.items()}, f"· escala MASE {escala_p4:.2f}")  # ← Anote (Tabla L: el modelo elegido, en su fila de P4)
res_p4.sort_values("RMSE")                              # ← Anote (Tabla L): el modelo elegido y la mejor línea base

In [ ]:
fig, ax = plt.subplots(figsize=(13, 4))
ax.plot(yt_p4.index, yt_p4.values, color=COLORES["real"], lw=1.2, label="real (prueba)")
ax.plot(yt_p4.index, preds_p4[elegido_p4], color=COLORES[elegido_p4], lw=1.0, label=f"{elegido_p4} (elegido en validación)")
ax.plot(yt_p4.index, preds_p4["Climatología estacional"], color="#165542", lw=0.8, ls="--", label="Climatología estacional")
ax.set(title="P4 · Serie propia: real frente a pronóstico en la prueba", xlabel="Fecha")
ax.legend(fontsize=8)
guardar(fig, "F16_serie_propia_real_vs_pronostico")  # → Figura F16 (reporte)
plt.show()

---
## P5 · Reproducción del artículo ancla

La especificación completa está en el **Anexo P** del manual: qué cifra o procedimiento del artículo ancla debe
reproducir, con qué datos y cómo reportar la discrepancia. Use las celdas siguientes para su implementación; la
evaluación se hace sobre su reporte, no sobre la coincidencia exacta de la cifra: lo que se califica es que la
discrepancia esté **medida y explicada**.

In [ ]:
# P5 · Su implementación (vea el Anexo P del manual).

In [ ]:
registrar("etapa6", {"tabla_J": {m: {f"{a}_{b}": round(float(v), 4) for (a, b), v in fila.items()}
                                  for m, fila in tabla_J.iterrows()},  # media y DE por método
                     "p1_duracion_s": round(t_p1, 1),
                     "p1_hiperparametros_por_pliegue": elegidos,
                     "p1_por_pliegue": cv.round(4).to_dict(orient="records"),
                     "tabla_K": tabla_K.round(5).to_dict(orient="records"),
                     "huecos_calendario_prueba": [f"{t:%Y-%m}" for t in huecos],
                     "p3": p3.round(3).to_dict(orient="records"), "p3_duracion_s": round(t_p3, 1),
                     "p3_resumen": resumen_p3.to_dict(orient="index"),
                     "p4_demostracion": RUTA_PROPIA is None, "p4_elegido_en_validacion": elegido_p4,
                     "p4_escala_mase": round(escala_p4, 4),
                     "p4": {n: {k: (v if isinstance(v, str) else round(float(v), 4)) for k, v in fila.items()}
                            for n, fila in res_p4.iterrows()}})
print("Registro de la Etapa 6 guardado en", (RES / "registro_P3.json").relative_to(BASE))

---
## Cierre de la Etapa 6

> **Punto de verificación.** Tiene las Tablas J (con su duración), K (14 filas) y L (con la duración de P3), el resumen de P3, la tabla de P4 y las figuras F14 a F16, y ha escrito la justificación de cada «Decisión metodológica» y de cada criterio de aceptación en su reporte.

Con esto termina la práctica. El reporte tipo artículo corto se describe en el Anexo P del manual.

In [ ]:
minutos = (time.perf_counter() - T0) / 60  # tiempo de pared desde el Bloque 0
registrar("etapa6_duracion_min", round(minutos, 2))
print(f"Etapa 6 terminada en {minutos:.1f} min · resultados en", RES.relative_to(BASE))  # ← Anote (Tabla A): duración